# Set A — M2-9, M2-11, M2-12, M2-13, M2-10 (duplicate-callback / idempotency)

**What this notebook does differently from the curl-replay runbook:** instead of replaying real HTTP
requests against the live `uvicorn` server (which means finding a real captured callback, checking its
bearer token hasn't expired, and hitting the actual `storage/` files), this notebook calls the exact same
real service functions **directly, in Python, in-process** — with all storage and outbound-ABDM calls
redirected to isolated, throwaway stand-ins. You can re-run any cell any number of times, tweak the
payload to try a new scenario, and never touch the running server, real ABDM, or anything under
`storage/`/`logs/`.

**What's real and what's stubbed, explicitly:**
- **Real:** the actual `process_*` service function from `server/callbacks/services/*.py` — the exact code
  path that runs when a real ABDM callback arrives. Nothing about the logic under test is faked.
- **Stubbed:** the storage location (`server/callbacks/utils/json_file_store.py`'s `_STORAGE_ROOT` is
  pointed at a fresh temp directory — see `harness.activate_scratch_storage()`), and every outbound call to
  ABDM (`send_on_consent_notify`, `send_on_confirm`, `notify_care_context_update`, `request_otp`,
  `send_on_init`) — replaced with an in-memory recorder (`harness.CallRecorder`) that returns a fake `202`
  and remembers how many times it was called and with what.

**Why this is safe to run any time:** `harness.py` never edits any file under `server/` or `tools/` — it
only imports and calls what's already there. The hard rule (never touch `storage/`/`logs/`) is enforced by
construction, not by discipline: the real repository code physically cannot see the real `storage/`
directory once `activate_scratch_storage()` has run, because the one variable it reads its file path from
has been repointed.

**Requirements:** run this from the repo root (or with the repo root on `sys.path`, see the first code
cell), using the project's own `.venv` as the Jupyter kernel — it needs `fastapi`, `cryptography`,
`requests`, etc. exactly like running the real server does, since it imports the real modules.

In [ ]:
import sys
from pathlib import Path

# This notebook lives at tools/edge_case_testing/notebooks/ -- repo root is 3 parents up.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "server").exists():
    # fall back to resolving relative to this notebook's own location
    REPO_ROOT = Path("__file__").resolve().parents[2] if Path("__file__").exists() else Path.cwd().parents[2]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("repo root on sys.path:", REPO_ROOT)
assert (REPO_ROOT / "server" / "callbacks").exists(), (
    "Couldn't find server/callbacks/ from here -- open this notebook with the repo root as the "
    "Jupyter working directory (File > Open Folder on the repo root), or edit REPO_ROOT above by hand."
)

import harness


---
## What is "idempotency" actually protecting against here? A real-world walkthrough

Before the mechanical per-case tests below, here's the concrete scenario that made M2-9 (and its siblings
M2-11/12/13) a real bug, not a theoretical one.

**The setup:** ABDM's gateway calls our `consent_notify` webhook whenever a patient's consent status
changes. Per that route's own docstring, this codebase fully processes the callback (verifies it, saves or
deletes the consent, etc.) *before* returning a 200 to ABDM. If that round trip is slow — a busy moment on
our server, a flaky network hop, anything — ABDM's gateway assumes the message didn't arrive and
**redelivers the exact same message again**, carrying the same `REQUEST-ID` as the first attempt.

**The real-world sequence that breaks without this fix:**
1. A patient opens the ABDM app and grants a hospital consent to view their records. ABDM calls our
   webhook: `status: GRANTED`, `REQUEST-ID: req-A`. Our server is a little slow to ack (nothing wrong,
   just a slow moment) — we still process it and save the consent, but the ack back to ABDM is delayed.
2. Because ABDM didn't get a fast ack for `req-A`, it queues that message for a retry — but hasn't sent it
   yet.
3. Seconds later, the same patient has second thoughts and revokes that same consent in the ABDM app. ABDM
   calls our webhook again: `status: REVOKED`, `REQUEST-ID: req-B` (a different message). We correctly
   delete the stored consent. From the hospital's side, that consent is now gone, as it should be.
4. **Then ABDM's queued retry of the ORIGINAL `req-A` GRANTED message finally arrives.** Without a replay
   guard, our code has no way to tell "this is the same GRANTED message I already handled" from "this is a
   brand-new grant" — it just sees `status: GRANTED` and saves the consent again. **The consent the patient
   deliberately revoked in step 3 is silently resurrected**, and the hospital's system now believes it still
   has valid, current consent to access data the patient explicitly said no to. That's a real privacy/
   compliance problem, not just a duplicate-processing annoyance.

**The fix:** `already_processed(scope, request_id)` remembers every `REQUEST-ID` this codebase has already
acted on, per callback type. When `req-A` comes back around in step 4, the guard recognizes it as a replay
of a message already handled and skips re-running the GRANTED branch entirely — the deliberately-revoked
consent stays deleted. ABDM still gets its ack either way (from ABDM's point of view, the message WAS
successfully handled), so this doesn't cause ABDM-side retries or errors either.

The cell below runs exactly this 3-step sequence against the real `process_consent_notify()` code and
proves the consent stays deleted after the replay.

In [ ]:
import asyncio
from unittest.mock import patch

import server.callbacks.services.consent_notify_service as consent_notify_service
from server.callbacks.repository.consent_repository import get_consent

harness.activate_scratch_storage("real_world_walkthrough")

send_on_consent_notify_recorder = harness.CallRecorder(harness.FakeResponse(202))

# Step 1: ABDM notifies us the patient GRANTED consent. REQUEST-ID req-A.
# (Imagine our ack back to ABDM was slow/lost here -- that's WHY ABDM will later retry this exact message.)
granted_message = {
    "headers": {"request-id": "req-A-original-grant"},
    "body": {
        "notification": {
            "consentId": "consent-real-world-walkthrough",
            "status": "GRANTED",
            "consentDetail": {
                "patient": {"id": "patient@sbx"},
                "careContexts": [{"referenceNumber": "cc-1", "careContextReference": "cc-1"}],
                "hiTypes": ["OPConsultation"],
                "permission": {"dateRange": {"from": "2026-01-01T00:00:00.000Z", "to": "2026-12-31T00:00:00.000Z"}},
                "hip": {"id": "IN2810000123"},
            },
        }
    },
}

# Step 3: the patient changes their mind seconds later and revokes. REQUEST-ID req-B -- a DIFFERENT,
# genuinely new message, not a replay of anything.
revoked_message = {
    "headers": {"request-id": "req-B-revoke"},
    "body": {"notification": {"consentId": "consent-real-world-walkthrough", "status": "REVOKED"}},
}

with patch.object(consent_notify_service, "send_on_consent_notify", send_on_consent_notify_recorder):
    print("== Step 1: ABDM notifies GRANTED (patient just consented) ==")
    await consent_notify_service.process_consent_notify(granted_message)
    print("consent stored right after grant?", get_consent("consent-real-world-walkthrough") is not None)

    print("\n== Step 3: patient revokes seconds later; ABDM notifies REVOKED (req-B, a new message) ==")
    await consent_notify_service.process_consent_notify(revoked_message)
    print("consent stored right after revoke?", get_consent("consent-real-world-walkthrough") is not None)

    print("\n== Step 4: ABDM's gateway retries the ORIGINAL req-A GRANTED message (its ack was slow the first time) ==")
    await consent_notify_service.process_consent_notify(granted_message)
    still_none = get_consent("consent-real-world-walkthrough") is None
    print("consent stored after the stale GRANTED replay?", not still_none)

print()
harness.check("the deliberately-revoked consent was NOT resurrected by the stale GRANTED replay", still_none)


---
## M2-9 — Consent Notify replay must not re-apply consent status

**Real code under test:** `server/callbacks/services/consent_notify_service.py` — `process_consent_notify()`.

**Scenario:** ABDM redelivers the exact same `consent_notify` callback (same `REQUEST-ID`) twice — this
happens for real whenever ABDM's gateway doesn't get a fast enough ack, per that module's own docstring.

**Must NOT happen twice:** the consent artefact must only be *saved* once — `save_consent()` must not run a
second time on the replay. What SHOULD still happen twice: the acknowledgement back to ABDM
(`send_on_consent_notify`) — a replay still gets acked, exactly as if it were processed successfully, per
the code's own comment.

**Pass criteria:** `send_on_consent_notify` called 2 times (both deliveries ack'd); exactly 1 consent record
ends up stored.

In [ ]:
import asyncio
from unittest.mock import patch

import server.callbacks.services.consent_notify_service as consent_notify_service
from server.callbacks.repository.consent_repository import get_all_consents

harness.activate_scratch_storage("m2_9")

send_on_consent_notify_recorder = harness.CallRecorder(harness.FakeResponse(202))

callback_data = {
    "headers": {"request-id": "req-m2-9-test-001"},
    "body": {
        "notification": {
            "consentId": "consent-abc-123",
            "status": "GRANTED",
            "consentDetail": {
                "patient": {"id": "patient@sbx"},
                "careContexts": [{"referenceNumber": "cc-1", "careContextReference": "cc-1"}],
                "hiTypes": ["OPConsultation"],
                "permission": {"dateRange": {"from": "2026-01-01T00:00:00.000Z", "to": "2026-12-31T00:00:00.000Z"}},
                "hip": {"id": "IN2810000123"},
            },
        }
    },
}

with patch.object(consent_notify_service, "send_on_consent_notify", send_on_consent_notify_recorder):
    print("--- 1st delivery ---")
    await consent_notify_service.process_consent_notify(callback_data)
    print("\n--- 2nd delivery (exact replay, same REQUEST-ID) ---")
    await consent_notify_service.process_consent_notify(callback_data)

print()
harness.check("send_on_consent_notify called exactly twice (both deliveries acked)",
              send_on_consent_notify_recorder.call_count == 2)
harness.check("exactly one consent record stored (not re-applied on replay)",
              len(get_all_consents()) == 1)


---
## M2-11 — Link Confirm replay must not re-verify OTP / re-send on-confirm

**Real code under test:** `server/callbacks/services/link_confirm_service.py` — `process_link_confirm()`.

**Setup needed:** this route looks up an existing link session (created by an earlier Link Init) and a
patient identity — both seeded here directly via the real repository save functions
(`save_link_session`, `save_patient_identity`), writing into the same isolated scratch storage.

**Stubbed:** `verify_otp` (so this doesn't need a real ABDM OTP), `search_patient` (so this doesn't depend
on `server/data/patient_records.csv` contents), `send_on_confirm`.

**Must NOT happen twice:** on replay, the whole body should be skipped — no second `send_on_confirm` call.

**Pass criteria:** `send_on_confirm` called exactly once, even though the service function itself was
called twice.

In [ ]:
import asyncio
from datetime import datetime, timezone, timedelta
from unittest.mock import patch

import server.callbacks.services.link_confirm_service as link_confirm_service
from server.callbacks.repository.link_repository import save_link_session
from server.callbacks.repository.patient_identity_repository import save_patient_identity

harness.activate_scratch_storage("m2_11")

save_link_session("LRN-TEST-001", {
    "transaction_id": "txn-1",
    "request_id": "orig-link-init-req",
    "abha_address": "test@sbx",
    "selected_patient_records": [],
    "otp_txn_id": "otp-txn-1",
    "otp_expiry": (datetime.now(timezone.utc) + timedelta(minutes=5)).isoformat(),
})
save_patient_identity("test@sbx", {"hip_id": "IN2810000123", "abha_number": "12-3456-7890-1234"})

send_on_confirm_recorder = harness.CallRecorder(harness.FakeResponse(202))

callback_data = {
    "headers": {"request-id": "req-m2-11-test-001"},
    "body": {"confirmation": {"token": "123456", "linkRefNumber": "LRN-TEST-001"}},
}

with patch.object(link_confirm_service, "verify_otp", lambda otp, txn_id: True), \
     patch.object(link_confirm_service, "search_patient", lambda **kw: []), \
     patch.object(link_confirm_service, "send_on_confirm", send_on_confirm_recorder):
    print("--- 1st delivery ---")
    await link_confirm_service.process_link_confirm(callback_data)
    print("\n--- 2nd delivery (exact replay, same REQUEST-ID) ---")
    await link_confirm_service.process_link_confirm(callback_data)

print()
harness.check("send_on_confirm called exactly once (not re-sent on replay)",
              send_on_confirm_recorder.call_count == 1)


---
## M2-12 — Care Context Link replay must not re-fire the Notify loop

**Real code under test:** `server/callbacks/services/care_context_link_service.py` —
`process_care_context_link()`.

**Setup needed:** a pending care-context-link session (created by the earlier outbound `link_care_context()`
call) — seeded via `save_pending_care_context_link()`. The real code deletes this record once it's consumed,
so it's re-seeded before the 2nd delivery purely so this test can isolate "does the idempotency guard alone
stop the 2nd delivery" from "there's nothing left to process anyway" — a genuine ABDM redelivery would
arrive as an exact duplicate before any deletion had a chance to matter in the first place; re-seeding here
just makes sure we're testing the guard, not a side effect of it having already cleaned up.

**Stubbed:** `notify_care_context_update` (would otherwise be a real outbound POST per care context).

**Must NOT happen twice:** the whole per-care-context Notify loop must not re-fire on replay.

**Pass criteria:** `notify_care_context_update` called exactly 2 times total (once per care context, only
on the 1st delivery — not 4 times).

In [ ]:
import asyncio
from unittest.mock import patch

import server.callbacks.services.care_context_link_service as care_context_link_service
from server.callbacks.repository.care_context_link_repository import save_pending_care_context_link

harness.activate_scratch_storage("m2_12")

pending_session = {
    "hip_id": "IN2810000123",
    "abha_address": "test@sbx",
    "patient_reference": "PAT-1",
    "link_token": "tok-1",
    "care_context_hi_types": {"cc-ref-1": ["OPConsultation"], "cc-ref-2": ["Prescription"]},
}
save_pending_care_context_link("req-m2-12-test-001", pending_session)

notify_recorder = harness.CallRecorder(harness.FakeResponse(202))

callback_data = {
    "body": {
        "abhaAddress": "test@sbx",
        "status": "SUCCESS",
        "response": {"requestId": "req-m2-12-test-001"},
    }
}

with patch.object(care_context_link_service, "notify_care_context_update", notify_recorder):
    print("--- 1st delivery ---")
    await care_context_link_service.process_care_context_link(callback_data)

    # Real code deletes the pending record after consuming it -- re-seed so this replay
    # exercises the idempotency guard itself, not "there's nothing left to do anyway."
    save_pending_care_context_link("req-m2-12-test-001", pending_session)

    print("\n--- 2nd delivery (exact replay, same REQUEST-ID) ---")
    await care_context_link_service.process_care_context_link(callback_data)

print()
harness.check("notify_care_context_update called exactly twice total (2 care contexts, 1st delivery only)",
              notify_recorder.call_count == 2)


---
## M2-13 — Link Init replay must not request a 2nd OTP / save a 2nd session

**Real code under test:** `server/callbacks/services/link_init_service.py` — `process_link_init()`.

**Setup needed:** a patient identity (seeded via `save_patient_identity`) — everything else this function
needs, it creates itself.

**Stubbed:** `request_otp`, `send_on_init` (both real outbound ABDM calls), and `encrypt_value` /
`get_public_certificate` (would otherwise try to fetch ABDM's real public certificate over the network to
encrypt the ABHA number).

**Must NOT happen twice:** no 2nd OTP request, no 2nd link session saved under a 2nd reference number.

**Pass criteria:** `request_otp` and `send_on_init` each called exactly once; exactly 1 link session
exists afterward.

In [ ]:
import asyncio
from unittest.mock import patch

import server.callbacks.services.link_init_service as link_init_service
from server.callbacks.repository.patient_identity_repository import save_patient_identity
from server.callbacks.repository.link_repository import get_all_link_sessions

harness.activate_scratch_storage("m2_13")

save_patient_identity("test@sbx", {"hip_id": "IN2810000123", "abha_number": "12-3456-7890-1234"})

request_otp_recorder = harness.CallRecorder(harness.FakeResponse(200, {"txnId": "otp-txn-xyz"}))
send_on_init_recorder = harness.CallRecorder(harness.FakeResponse(202))

callback_data = {
    "headers": {"request-id": "req-m2-13-test-001"},
    "body": {"abhaAddress": "test@sbx", "transactionId": "txn-init-1", "patient": []},
}

with patch.object(link_init_service, "request_otp", request_otp_recorder), \
     patch.object(link_init_service, "send_on_init", send_on_init_recorder), \
     patch.object(link_init_service, "encrypt_value", lambda val, cert: f"ENCRYPTED:{val}"), \
     patch.object(link_init_service, "get_public_certificate", lambda: "FAKE_CERT"):
    print("--- 1st delivery ---")
    await link_init_service.process_link_init(callback_data)
    print("\n--- 2nd delivery (exact replay, same REQUEST-ID) ---")
    await link_init_service.process_link_init(callback_data)

print()
harness.check("request_otp called exactly once (no 2nd OTP requested on replay)",
              request_otp_recorder.call_count == 1)
harness.check("send_on_init called exactly once", send_on_init_recorder.call_count == 1)
harness.check("exactly one link session saved (not a 2nd one under a new reference number)",
              len(get_all_link_sessions()) == 1)


---
## M2-10 — Health Information Request replay must not re-encrypt/re-push under new keys

**Real code under test:** `server/callbacks/services/health_information_request_service.py` —
`process_health_information_request()`. Same `idempotency.py` module as M2-9/11/12/13, wired into a fifth
service.

**Real-world scenario:** `_push_and_notify()` generates a fresh ECDH key pair per care context on every
call, by design, for forward secrecy. A raw replay of this callback with no guard would re-encrypt and
re-push every record under brand-new keys, and re-notify ABDM a second time for a transfer that already
completed — the same "data sent out twice, under two different encryption keys" problem, one service over.

**Must NOT happen twice:** the whole encrypt→push→notify cycle. What SHOULD still happen twice: the
on-request ack.

**Pass criteria:** `send_on_health_information_request` called twice (both deliveries acked);
`send_health_information_notify` called only once (no 2nd cycle).

In [ ]:
import asyncio
from unittest.mock import patch

import server.callbacks.services.health_information_request_service as health_information_request_service

harness.activate_scratch_storage("m2_10")

send_on_request_recorder = harness.CallRecorder(harness.FakeResponse(200))
send_notify_recorder = harness.CallRecorder(harness.FakeResponse(202))

# No consent seeded for this consentId on purpose -- this keeps the test focused purely on the
# idempotency guard itself (fhir_bundles ends up None either way, taking the short "notify FAILED"
# path) rather than needing the full encrypt/push machinery, which M2-6's notebook already covers.
callback_data = {
    "headers": {"request-id": "req-m2-10-test-001", "x-hip-id": "IN2810000123"},
    "body": {"transactionId": "txn-m2-10", "hiRequest": {"consent": {"id": "consent-never-granted"}, "dataPushUrl": None, "keyMaterial": {}}},
}

with patch.object(health_information_request_service, "send_on_health_information_request", send_on_request_recorder), \
     patch.object(health_information_request_service, "send_health_information_notify", send_notify_recorder):
    print("--- 1st delivery ---")
    await health_information_request_service.process_health_information_request(callback_data)
    print("\n--- 2nd delivery (exact replay, same REQUEST-ID) ---")
    await health_information_request_service.process_health_information_request(callback_data)

print()
harness.check("send_on_health_information_request called exactly twice (both deliveries acked)",
              send_on_request_recorder.call_count == 2)
harness.check("send_health_information_notify called exactly once (no 2nd encrypt+push+notify cycle)",
              send_notify_recorder.call_count == 1)


---
## Bonus pattern — corrupted / malformed values, same reusable shape

You mentioned wanting to test "give a corrupt value" scenarios too, not just duplicates — the exact same
pattern covers that. You don't touch any source file: just mutate the `callback_data` dict in a notebook
cell before calling the real service function. Two examples below against `process_consent_notify` —
neither should crash (the real code wraps its body in `try/except` and logs an error instead), and neither
should store anything.

In [ ]:
import asyncio
from unittest.mock import patch

import server.callbacks.services.consent_notify_service as consent_notify_service
from server.callbacks.repository.consent_repository import get_all_consents

harness.activate_scratch_storage("m2_9_corrupt")

send_on_consent_notify_recorder = harness.CallRecorder(harness.FakeResponse(202))

# Corrupt value #1: notification present but consentId missing entirely.
corrupt_missing_consent_id = {
    "headers": {"request-id": "req-corrupt-001"},
    "body": {"notification": {"status": "GRANTED", "consentDetail": {}}},
}

# Corrupt value #2: notification key itself missing (totally malformed body).
corrupt_missing_notification = {
    "headers": {"request-id": "req-corrupt-002"},
    "body": {"somethingElse": True},
}

with patch.object(consent_notify_service, "send_on_consent_notify", send_on_consent_notify_recorder):
    print("--- corrupt payload 1: missing consentId ---")
    await consent_notify_service.process_consent_notify(corrupt_missing_consent_id)
    print("\n--- corrupt payload 2: missing notification key entirely ---")
    await consent_notify_service.process_consent_notify(corrupt_missing_notification)

print()
harness.check("neither corrupt payload crashed the notebook kernel (both handled gracefully)", True)
harness.check("nothing was stored from either corrupt payload",
              len(get_all_consents()) == 0)
harness.check("ABDM was never acked for either corrupt payload (no consentId to ack with)",
              send_on_consent_notify_recorder.call_count == 0)


---
## Summary

If every `PASS` line above printed `PASS`, all 4 of M2-9 / M2-11 / M2-12 / M2-13 are verified against the
real fix code. Send me (or paste back) the printed output from each cell and I'll move those Notion rows to
Done. Anything that prints `FAIL` goes back to Not started for re-diagnosis, per the 3-stage rule — tell me
which cell and I'll take a look.

**To build a similar notebook for another batch:** copy this notebook's structure — one markdown cell
explaining the scenario and pass criteria, one code cell that seeds any needed state, patches the outbound
calls with `harness.CallRecorder`, builds a `callback_data` dict, and calls the real `process_*()` function
via `await process_*(callback_data)` (Jupyter supports top-level `await` directly in a code cell) as many
times/with whatever mutations the scenario needs, ending in `harness.check(...)` calls. `harness.py` itself
doesn't need to change for a new case unless it needs a genuinely new capability.